# Evaluation Notebook (RTN + AWQ)

This notebook evaluates the following models and saves results into their corresponding
`quantized_models/<model>/` directories:

- `baseline_fp16` (base model)
- `rtn_w8`
- `rtn_w4`
- `awq_w4`

Each model's results are saved as both JSON and CSV (`evaluation.json` / `evaluation.csv`).

> **Note:** Run this notebook in the environment that supports `awq` / `rtn` evaluation (e.g., `awq-env`).

In [1]:
# # Install any missing packages (run once)
# # You can comment this cell out once dependencies are satisfied.

# import sys
# import subprocess

# REQUIRED = [
#     "torch",
#     "transformers",
#     "datasets",
#     "accelerate",
#     "awq",
#     "pandas",
#     "tqdm",
# ]

# for pkg in REQUIRED:
#     try:
#         __import__(pkg)
#     except ImportError:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# print("Dependencies verified.")

In [2]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import utils.model_loader as model_loader
import utils.eval_utils as eval_utils

# Where quantized models are stored
OUTPUT_ROOT = Path("./quantized_models")

# Configuration: list of models to evaluate
MODEL_CONFIGS = [
    # {"name": "baseline_fp16", "path": model_loader.MODEL_ID, "bits": 16, "run_lm_harness": True},
    # {"name": "rtn_w8", "path": str(OUTPUT_ROOT / "rtn_w8"), "bits": 8, "run_lm_harness": True},
    # {"name": "rtn_w4", "path": str(OUTPUT_ROOT / "rtn_w4"), "bits": 4, "run_lm_harness": True},
    {"name": "awq_w4", "path": str(OUTPUT_ROOT / "awq_w4"), "bits": 4, "run_lm_harness": True},
]

# Perplexity evaluation settings
PERPLEXITY_SAMPLES = 100
MAX_LENGTH = 512


In [3]:
def load_and_prepare_model(config: dict, device: str = "cuda"):
    """Load a model and tokenizer for evaluation."""

    model_path = config["path"]
    model_name = config["name"]


    print("Model Path: ", model_path)

    if "awq" in model_name.lower():
        # AutoAWQ supports quantized model loading via `from_quantized`
        from awq import AutoAWQForCausalLM

        model = AutoAWQForCausalLM.from_quantized(model_path, fuse_layers=True)

    elif "baseline" in model_name.lower():
        # Load from HF cachel
        model = model_loader.load_base_model()  # uses MODEL_ID by default

    else: 
        # RTN models 
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto",
        )

    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)


    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer



In [4]:
# for cfg in MODEL_CONFIGS:
#     model, tokenizer = load_and_prepare_model(cfg)

#     lm_metrics = {}
#     if cfg.get("run_lm_harness", False):
#         lm_metrics = eval_utils.evaluate_lm_harness(
#             model_obj=model,
#             tokenizer_obj=tokenizer,
#             # Updated tasks to specific MMLU groups
#             tasks=["hellaswag", "piqa", "mmlu_stem", "mmlu_humanities"],
#             num_fewshot=3,
#             limit=30,
#             max_gen_toks=256
#         )
        
#     print(lm_metrics)

In [5]:
# import matplotlib.pyplot as plt
# import pandas as pd
# import seaborn as sns

# def plot_lm_eval_results(eval_output, model_name="Model"):
#     res_dict = eval_output['results']
    
#     data = []
#     for task, metrics in res_dict.items():
#         # Map messy internal names to pretty labels
#         display_name = {
#             "mmlu_stem": "MMLU Stem (Avg)",
#             "mmlu_humanities": "MMLU Humanities (Avg)",
#             "hellaswag": "HellaSwag",
#             "piqa": "PIQA"
#         }.get(task, task)

#         data.append({
#             "Task Group": display_name,
#             "Accuracy": metrics.get("acc,none") or metrics.get("acc") or 0,
#             # "Acc (Norm)": metrics.get("acc_norm,none") or metrics.get("acc_norm") or 0
#         })
    
#     df_plot = pd.DataFrame(data)
#     df_melted = df_plot.melt(id_vars="Task Group", var_name="Metric", value_name="Value")
    
#     plt.figure(figsize=(10, 6))
#     sns.set_style("whitegrid")
#     ax = sns.barplot(data=df_melted, x="Task Group", y="Value", hue="Metric", palette="viridis")
    
#     plt.title(f"Evaluation Results for {model_name}", fontsize=14, pad=15)
#     plt.ylim(0, 1.0)
#     plt.tight_layout()
#     plt.show()

# # Run it
# # 2. STRICT FILTERING: Only keep the 4 main bars you want
# main_tasks = ["hellaswag", "piqa", "mmlu_stem", "mmlu_humanities"]
        
# # We create a new dict that ONLY contains these 4 keys
# filtered_metrics = {
#     "results": {k: v for k, v in lm_metrics['results'].items() if k in main_tasks}
# }
# plot_lm_eval_results(filtered_metrics, model_name=cfg['name'])

In [6]:
def evaluate_model_config(config: dict):
    """Evaluate a model and save the results into its model directory."""

    print("\n" + "=" * 80)
    print(f"Evaluating: {config['name']}")
    print("=" * 80)

    model_dir = config["path"]
    model_name = config["name"]

    model, tokenizer = load_and_prepare_model(config)

    # 1) Size/Bandwidth estimate
    size_metrics = eval_utils.calculate_model_size_and_bandwidth(
        model, bits=config.get("bits", 16)
    )

    # 2) Perplexity
    ppl_metrics = eval_utils.evaluate_perplexity(
        model,
        tokenizer,
        max_samples=PERPLEXITY_SAMPLES,
        max_length=MAX_LENGTH,
        device="cuda"
    )

    # 3) Optional LM Evaluation Harness (can take longer)
    lm_metrics = {}
    if config.get("run_lm_harness", False):
        if "awq" in model_name.lower():
            model = getattr(model, "model", model)

        lm_metrics = eval_utils.evaluate_lm_harness(
            model_obj=model,
            tokenizer_obj=tokenizer,
            # Updated tasks to specific MMLU groups
            tasks=["hellaswag", "piqa", "mmlu_stem", "mmlu_humanities"],
            num_fewshot=3,
            limit=5,
        )

    # Merge metrics
    results = {
        "model_name": config["name"],
        "model_path": str(model_dir),
        "bits": config.get("bits", 16),
        **size_metrics,
        **ppl_metrics,
        "lm_eval": lm_metrics,
    }

    # Save per-model results
    save_dir = model_dir
    if "baseline" in config["name"]:
        save_dir = str(OUTPUT_ROOT / "baseline")

    eval_utils.save_eval_result(results, save_dir, file_stem="evaluation")

    # Cleanup
    del model
    torch.cuda.empty_cache()

    print(f"Saved evaluation to: evaluation.json")
    return results


# Run evaluation for all configured models
all_results = []
for cfg in MODEL_CONFIGS:
    try:
        results = evaluate_model_config(cfg)
        all_results.append(results)
    except Exception as e:
        print(f"Failed to evaluate {cfg['name']}: {e}")
        raise ValueError

# Save combined results (optional)
combined_dir = OUTPUT_ROOT / "analysis"
combined_dir.mkdir(parents=True, exist_ok=True)

import pandas as pd

if all_results:
    df = pd.DataFrame(all_results)
    df.to_csv(combined_dir / "evaluation_awq_rtn_results.csv", index=False)
    df.to_json(combined_dir / "evaluation_awq_rtn_results.json", orient="records", indent=2)
    print(f"\nSaved combined results to: {combined_dir}")



Evaluating: awq_w4
Model Path:  quantized_models/awq_w4


/mnt/miniconda3/envs/awq-env/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/mnt/miniconda3/envs/awq-env/lib/python3.10/site-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
`t

Evaluating perplexity on wikitext test split (100 samples)...


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 19.81it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of mmlu_formal_logic from None to 3
Overwriting default num_fewshot of mmlu_high_school_european_history from None to 3
Overwriting default num_fewshot of mmlu_high_school_us_history from None to 3
Overwriting default num_fewshot of mmlu_high_school_world_history from None to 3
Overwriting default num_fewshot of mmlu_international_law from None to 3
Overwriting default num_fewshot of mmlu_jurisprudence from None to 3
Overwriting default num_fewshot of mmlu_logical_fallacies from None to 3
Overwriting default num_fewshot of mmlu_moral_disputes from None to 3
Overwriting defa

Saved evaluation to: evaluation.json

Saved combined results to: quantized_models/analysis
